# 01 - Data Cleaning

## Scope
This notebook documents data preparation decisions for the Zurich transport delay dataset and validates the final cleaned output used in downstream analysis.

## Inputs and Outputs
- Input: `../data/raw/vbz_delays_original.csv`
- Outputs: `../data/processed/vbz_delays_clean.csv`, `../data/processed/vbz_delays.db`

In [9]:
from pathlib import Path
import pandas as pd

raw_path = Path('../data/raw/vbz_delays_original.csv')
clean_path = Path('../data/processed/vbz_delays_clean.csv')
db_path = Path('../data/processed/vbz_delays.db')
raw_path, clean_path, db_path

(WindowsPath('../data/raw/vbz_delays_original.csv'),
 WindowsPath('../data/processed/vbz_delays_clean.csv'),
 WindowsPath('../data/processed/vbz_delays.db'))

## Raw Data Snapshot

In [10]:
df_raw_sample = pd.read_csv(raw_path, nrows=5000, low_memory=False)
df_raw_sample.head()

,linie,richtung,betriebsdatum,fahrzeug,kurs,seq_von,halt_diva_von,halt_punkt_diva_von,halt_kurz_von1,datum_von,...,fahrweg_id,fw_no,fw_typ,fw_kurz,fw_lang,umlauf_von,halt_id_von,halt_id_nach,halt_punkt_id_von,halt_punkt_id_nach
0,12,2,31.12.22,3062,21,13,994,1,BGLA07,31.12.22,...,160499,4,1,4,BSTE - FRAF Mutation auf Li 10,291491,1452,1378,42904,45988
1,12,2,31.12.22,3066,34,13,994,1,BGLA07,31.12.22,...,160499,4,1,4,BSTE - FRAF Mutation auf Li 10,290466,1452,1378,42904,45988
2,12,2,31.12.22,3071,37,13,994,1,BGLA07,31.12.22,...,160499,4,1,4,BSTE - FRAF Mutation auf Li 10,290337,1452,1378,42904,45988
3,12,2,31.12.22,3071,37,13,994,1,BGLA07,31.12.22,...,160497,2,1,2,BSTE - FRAF,290337,1452,1378,42904,45988
4,12,2,31.12.22,3072,25,13,994,1,BGLA07,31.12.22,...,160499,4,1,4,BSTE - FRAF Mutation auf Li 10,291709,1452,1378,42904,45988


In [11]:
df_raw_sample.shape

(5000, 34)

## Cleaning Logic Implemented in `src/cleaning.py`
- Standardized selected column names
- Parsed service dates and converted timing fields to numeric types
- Removed rows with critical missing timing/date fields
- Engineered delay and travel-time metrics
- Loaded cleaned table into SQLite and created analysis indexes

## Clean Data Validation

In [12]:
df_clean = pd.read_csv(clean_path)
df_clean.head()

,linie,betriebsdatum,halt_kurz_von,halt_kurz_nach,soll_ab_von,ist_ab_von,soll_an_nach,ist_an_nach,departure_hour,departure_delay_sec,arrival_delay_sec,scheduled_travel_sec,actual_travel_sec,travel_time_diff_sec,is_delayed_departure
0,12,2022-12-31,BGLA07,RBAU07,44538,44519,44616,44597,12,-19,-19,78,78,0,0
1,12,2022-12-31,BGLA07,RBAU07,60738,60763,60816,60847,16,25,31,78,84,6,1
2,12,2022-12-31,BGLA07,RBAU07,33636,33659,33714,33741,9,23,27,78,82,4,1
3,12,2022-12-31,BGLA07,RBAU07,76836,76837,76914,76912,21,1,-2,78,75,-3,1
4,12,2022-12-31,BGLA07,RBAU07,36336,36482,36414,36561,10,146,147,78,79,1,1


In [13]:
df_clean.shape

(1402968, 15)

In [14]:
df_clean.isna().mean().sort_values(ascending=False).head(10)

linie                  0.0
betriebsdatum          0.0
halt_kurz_von          0.0
halt_kurz_nach         0.0
soll_ab_von            0.0
ist_ab_von             0.0
soll_an_nach           0.0
ist_an_nach            0.0
departure_hour         0.0
departure_delay_sec    0.0
dtype: float64

## Data Dictionary (Core Features)
- `linie`: transport line identifier
- `betriebsdatum`: service date
- `departure_delay_sec`: departure delay in seconds
- `arrival_delay_sec`: arrival delay in seconds
- `scheduled_travel_sec`: scheduled segment travel time
- `actual_travel_sec`: observed segment travel time
- `travel_time_diff_sec`: actual minus scheduled travel time
- `departure_hour`: scheduled departure hour